# NAM — Neural Additive Models (L1, L2, L3)

**Arquitectura:** un MLP independiente por feature → suma de contribuciones → logit
**Ventaja clave:** cada MLP aprende una *shape function* interpretable para su variable
**Features:** 8 numéricas — 6 vitales de triage + `acuity` (ESI) + `n_medications`
**Nota:** `chiefcomplaint` se excluye porque sus ~43k categorías únicas no se mapean a una función escalar 1D interpretable.
**Targets:** L1 (ingreso hospitalario), L2 (resultado crítico), L3 (intervención crítica)

> Referencia: Agarwal et al. (2021) *Neural Additive Models: Interpretable Machine Learning with Neural Nets*. NeurIPS 2021.

El NAM se incluye como modelo de interpretabilidad: su estructura aditiva permite extraer directamente la contribución marginal de cada variable al riesgo predicho, lo que facilita la validación clínica de las relaciones aprendidas.

## 1. Setup

Importaciones y configuración de reproducibilidad. Se fijan semillas en NumPy y PyTorch. El device se selecciona automáticamente (CUDA si está disponible, CPU en caso contrario).

In [1]:
import os, json, pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

Device: cpu


## 2. Carga de datos

Se cargan las particiones de train y validación. Se añade `n_medications` desde `medrecon` mediante left join; las estancias sin registro reciben valor 0.

In [2]:
load_dotenv(dotenv_path=Path("../../.env"), override=True)
if not os.getenv("MIMIC_IV_ED_PATH"):
    load_dotenv(dotenv_path=Path(".env"), override=True)

DATA          = Path(os.getenv("MIMIC_IV_ED_PATH", ""))
PROCESSED_DIR = Path("../../data/processed")

df_train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
df_val   = pd.read_parquet(PROCESSED_DIR / "val.parquet")

print(f"Train : {df_train.shape[0]:,} pacientes, {df_train.shape[1]} columnas")
print(f"Val   : {df_val.shape[0]:,} pacientes, {df_val.shape[1]} columnas")

Train : 278,320 pacientes, 21 columnas
Val   : 59,640 pacientes, 21 columnas


## 3. n_medications (proxy de polifarmacia)

Conteo de medicamentos previos por estancia (`stay_id`) a partir de `medrecon`. Se utiliza como proxy de comorbilidad y complejidad clínica del paciente al ingreso.

In [3]:
df_medrecon = pd.read_csv(DATA / "medrecon.csv", low_memory=False)
n_meds = df_medrecon.groupby("stay_id").size().rename("n_medications").reset_index()

df_train = df_train.merge(n_meds, on="stay_id", how="left")
df_val   = df_val.merge(n_meds, on="stay_id", how="left")
df_train["n_medications"] = df_train["n_medications"].fillna(0).astype(float)
df_val["n_medications"]   = df_val["n_medications"].fillna(0).astype(float)

print(f"n_medications — media train: {df_train['n_medications'].mean():.2f}, mediana: {df_train['n_medications'].median():.0f}")

n_medications — media train: 6.96, mediana: 4


## 4. Preprocesamiento: imputación + escalado

Imputación con la mediana y escalado Z-score ajustados exclusivamente sobre train para evitar data leakage. Se normalizan las 8 features numéricas antes de pasarlas a cada `FeatureNet`.

In [ ]:
FEATURES = ["temperature", "heartrate", "resprate", "o2sat", "sbp", "dbp", "acuity", "n_medications", "pain"]
TARGETS  = ["L1", "L2", "L3"]

# Ajuste exclusivo sobre train para evitar data leakage
imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(df_train[FEATURES].values))
X_val   = scaler.transform(imputer.transform(df_val[FEATURES].values))

print(f"Features ({len(FEATURES)}): {FEATURES}")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")

## 5. Dataset

`TriageDataset` adapta los arrays NumPy al formato esperado por `DataLoader`, devolviendo pares `(X, y)` como tensores float.

In [5]:
class TriageDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## 6. Arquitectura NAM

```
x₁ → MLP₁ → f₁(x₁) ─┐
x₂ → MLP₂ → f₂(x₂) ─┤
...                   ├→ Σ + bias → logit → sigmoid → P(outcome)
x₈ → MLP₈ → f₈(x₈) ─┘
```

Cada `fᵢ` es independiente: su gráfica (valor de entrada vs. contribución al logit) es la *shape function* interpretable.

In [6]:
class FeatureNet(nn.Module):
    """MLP para una sola feature — aprende su shape function."""
    def __init__(self, hidden=64, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):  # x: (B,)
        return self.net(x.unsqueeze(-1)).squeeze(-1)  # (B,)


class NAM(nn.Module):
    """Neural Additive Model: logit = bias + Σ fᵢ(xᵢ)"""
    def __init__(self, n_features, hidden=64, dropout=0.1):
        super().__init__()
        self.nets = nn.ModuleList([FeatureNet(hidden, dropout) for _ in range(n_features)])
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):  # x: (B, n_features)
        contrib = torch.stack([net(x[:, i]) for i, net in enumerate(self.nets)], dim=1)
        return contrib.sum(dim=1) + self.bias  # (B,)

    @torch.no_grad()
    def get_contributions(self, x):  # → (B, n_features)
        return torch.stack([net(x[:, i]) for i, net in enumerate(self.nets)], dim=1)


# Verificación rápida de dimensiones
_dummy = torch.randn(4, len(FEATURES))
_model = NAM(n_features=len(FEATURES))
assert _model(_dummy).shape == (4,), "forward shape error"
assert _model.get_contributions(_dummy).shape == (4, len(FEATURES)), "contributions shape error"
print(f"NAM OK — parámetros totales: {sum(p.numel() for p in _model.parameters()):,}")

NAM OK — parámetros totales: 34,825


## 7. Entrenamiento

Bucle de entrenamiento con early stopping sobre AUROC en validación. Se emplea `BCEWithLogitsLoss` con `pos_weight` dinámico y `ReduceLROnPlateau` para adaptar la tasa de aprendizaje durante el entrenamiento.

In [7]:
def train_nam(X_tr, y_tr, X_v, y_v,
              hidden=64, dropout=0.1, lr=1e-3, wd=1e-4,
              batch_size=2048, max_epochs=50, patience=7):

    pw        = torch.tensor([(1 - y_tr.mean()) / y_tr.mean()], dtype=torch.float).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    dl_tr = DataLoader(TriageDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    dl_v  = DataLoader(TriageDataset(X_v,  y_v),  batch_size=batch_size * 4)

    model = NAM(n_features=X_tr.shape[1], hidden=hidden, dropout=dropout).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=3, factor=0.5)

    best_auroc, best_state, wait = 0.0, None, 0

    for epoch in range(1, max_epochs + 1):
        # Entrenamiento
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(xb), yb).backward()
            opt.step()

        # Validación
        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for xb, yb in dl_v:
                preds.append(torch.sigmoid(model(xb.to(DEVICE))).cpu())
                labels.append(yb)
        preds  = torch.cat(preds).numpy()
        labels = torch.cat(labels).numpy()
        auroc  = roc_auc_score(labels, preds)
        sched.step(auroc)

        # Early stopping
        if auroc > best_auroc:
            best_auroc = auroc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop en epoch {epoch} — mejor AUROC val: {best_auroc:.4f}")
                break

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | AUROC val: {auroc:.4f} | lr: {opt.param_groups[0]['lr']:.2e}")

    model.load_state_dict(best_state)
    return model

In [ ]:
results        = {}
trained_models = {}

for target in TARGETS:
    y_tr = df_train[target].values.astype(float)
    y_v  = df_val[target].values.astype(float)

    print(f"\n{'='*55}")
    print(f"TARGET: {target}  (prevalencia val: {y_v.mean()*100:.2f}%)")
    print(f"{'='*55}")

    model = train_nam(X_train, y_tr, X_val, y_v)

    # Métricas finales sobre validación
    model.eval()
    preds_all = []
    with torch.no_grad():
        for xb, _ in DataLoader(TriageDataset(X_val, y_v), batch_size=8192):
            preds_all.append(torch.sigmoid(model(xb.to(DEVICE))).cpu())
    preds_all = torch.cat(preds_all).numpy()

    auroc = roc_auc_score(y_v, preds_all)
    auprc = average_precision_score(y_v, preds_all)
    brier = brier_score_loss(y_v, preds_all)

    print(f"  AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | Brier: {brier:.4f}")
    results[target]        = {"AUROC": auroc, "AUPRC": auprc, "Brier": brier, "Prev_val": y_v.mean()}
    trained_models[target] = model

print("\nNAM entrenado para L1, L2 y L3.")

## 8. Tabla comparativa de resultados

El NAM queda por debajo de CatBoost y LSTM en discriminación (AUROC), resultado esperado dado que opera únicamente sobre 8 features numéricas y sin interacciones entre variables. Su valor principal no es predictivo sino interpretativo: las shape functions de la sección siguiente permiten validar la coherencia clínica de las relaciones aprendidas.

In [9]:
prev_results = {
    "LogReg+LASSO":  {"L1": (0.8048, 0.7102), "L2": (0.8617, 0.0962), "L3": (0.8773, 0.0705)},
    "CatBoost":      {"L1": (0.8314, 0.7523), "L2": (0.8795, 0.2228), "L3": (0.8943, 0.1141)},
    "LSTM":          {"L1": (0.7905, 0.7056), "L2": (0.8732, 0.1251), "L3": (0.8987, 0.1325)},
    "TabTransformer":{"L1": (0.8099, 0.7192), "L2": (0.8571, 0.1056), "L3": (0.8638, 0.0667)},
}

rows = []
for name, res in prev_results.items():
    rows.append({"Modelo": name,
                 **{f"{t} AUROC": res[t][0] for t in TARGETS},
                 **{f"{t} AUPRC": res[t][1] for t in TARGETS}})

rows.append({"Modelo": "NAM",
             **{f"{t} AUROC": results[t]["AUROC"] for t in TARGETS},
             **{f"{t} AUPRC": results[t]["AUPRC"] for t in TARGETS}})

df_comp = pd.DataFrame(rows).set_index("Modelo")
print("=== Comparativa completa — validación (AUROC | AUPRC) ===")
print(df_comp.round(4).to_string())

=== Comparativa completa — validación (AUROC | AUPRC) ===
                L1 AUROC  L2 AUROC  L3 AUROC  L1 AUPRC  L2 AUPRC  L3 AUPRC
Modelo                                                                    
LogReg+LASSO      0.8048    0.8617    0.8773    0.7102    0.0962    0.0705
CatBoost          0.8314    0.8795    0.8943    0.7523    0.2228    0.1141
LSTM              0.7905    0.8732    0.8987    0.7056    0.1251    0.1325
TabTransformer    0.8099    0.8571    0.8638    0.7192    0.1056    0.0667
NAM               0.7762    0.8332    0.8713    0.6761    0.0757    0.0820


## 9. Shape Functions — interpretabilidad

Para cada feature se barre su rango observado en train y se grafica la contribución aprendida al logit. Una curva ascendente indica que valores altos de esa variable aumentan el riesgo predicho; descendente, lo reducen.

In [ ]:
FEATURE_LABELS = {
    "temperature":   "Temperatura (°C)",
    "heartrate":     "Frecuencia cardíaca (lpm)",
    "resprate":      "Frecuencia respiratoria (rpm)",
    "o2sat":         "SpO₂ (%)",
    "sbp":           "PAS (mmHg)",
    "dbp":           "PAD (mmHg)",
    "acuity":        "Acuity / ESI",
    "n_medications": "N.º medicamentos",
    "pain":          "Dolor EVA (0-10)",
}

fig, axes = plt.subplots(3, len(FEATURES), figsize=(24, 10), sharey="row")

for row, target in enumerate(TARGETS):
    model = trained_models[target]
    model.eval()

    for col, feat in enumerate(FEATURES):
        ax = axes[row, col]

        grid_scaled = np.linspace(X_train[:, col].min(), X_train[:, col].max(), 300)

        with torch.no_grad():
            contrib = model.nets[col](
                torch.FloatTensor(grid_scaled).to(DEVICE)
            ).cpu().numpy()

        orig = grid_scaled * scaler.scale_[col] + scaler.mean_[col]

        ax.plot(orig, contrib, color="#1565C0", lw=1.5)
        ax.axhline(0, color="gray", lw=0.7, ls="--")
        ax.set_title(FEATURE_LABELS[feat] if row == 0 else "", fontsize=7.5, pad=4)
        ax.set_xlabel("Valor", fontsize=7)
        if col == 0:
            ax.set_ylabel(f"{target}\nContrib. logit", fontsize=8)
        ax.tick_params(labelsize=6.5)

plt.suptitle("NAM — Shape Functions por feature y target", y=1.01, fontsize=12, fontweight="bold")
plt.tight_layout()

FIG_DIR = Path("../../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FIG_DIR / "nam_shape_functions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura guardada en {FIG_DIR / 'nam_shape_functions.png'}")

## 10. Guardado de artefactos

Se persisten los pesos del modelo (`.pt`), las métricas, la configuración arquitectónica y los preprocesadores (imputador y escalador) en `models/nam/<timestamp>/` para su uso en la generación de shape functions finales y en la evaluación sobre test.

In [ ]:
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR  = Path(f"../../models/nam/{TIMESTAMP}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for target, model in trained_models.items():
    torch.save(model.state_dict(), SAVE_DIR / f"nam_{target}.pt")

with open(SAVE_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

with open(SAVE_DIR / "model_config.json", "w") as f:
    json.dump({"features": FEATURES, "hidden": 64, "dropout": 0.1, "n_features": len(FEATURES)}, f, indent=2)

with open(SAVE_DIR / "preprocessors.pkl", "wb") as f:
    pickle.dump({"imputer": imputer, "scaler": scaler}, f)

print(f"Artefactos guardados en {SAVE_DIR}")
print(f"   Modelos: nam_L1.pt, nam_L2.pt, nam_L3.pt")
print(f"   Configs: results.json, model_config.json, preprocessors.pkl")